# Create Mini HDF5 Cache

Build shared mini CLIP patch caches from the full HDF5 backup before running EXP notebooks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import random
import shutil
from pathlib import Path

import h5py
from tqdm.auto import tqdm

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/blip2_project')
FULL_CACHE_DIR = DATA_ROOT / 'cache_backup'
MINI_CACHE_DIR = DATA_ROOT / 'cache'
LOCAL_OUT = Path('/content/blip2_mini_cache_build')
SEED = 42
TRAIN_IMAGES = 20_000
VAL_IMAGES = 10_000
FORCE_REBUILD = False

SOURCE_H5 = {
    'train': FULL_CACHE_DIR / 'train_features.h5',
    'val': FULL_CACHE_DIR / 'val_features.h5',
}
DRIVE_MINI_H5 = {
    'train': MINI_CACHE_DIR / 'train_features_mini.h5',
    'val': MINI_CACHE_DIR / 'val_features_mini.h5',
}
LOCAL_MINI_H5 = {split: LOCAL_OUT / path.name for split, path in DRIVE_MINI_H5.items()}
MANIFEST_PATH = MINI_CACHE_DIR / 'mini_cache_20k_10k_manifest.json'
SPLIT_SIZES = {'train': TRAIN_IMAGES, 'val': VAL_IMAGES}

for path in SOURCE_H5.values():
    if not path.exists():
        raise FileNotFoundError(f'Missing full backup H5: {path}')
MINI_CACHE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
print('Full cache:', FULL_CACHE_DIR)
print('Mini cache output:', MINI_CACHE_DIR)

In [ ]:
def count_h5_keys(path):
    with h5py.File(path, 'r') as h5f:
        return len(h5f.keys())

def existing_outputs_are_valid():
    if FORCE_REBUILD or not MANIFEST_PATH.exists():
        return False
    if not all(path.exists() for path in DRIVE_MINI_H5.values()):
        return False
    manifest = json.loads(MANIFEST_PATH.read_text())
    if manifest.get('seed') != SEED or manifest.get('split_sizes') != SPLIT_SIZES:
        return False
    return all(count_h5_keys(DRIVE_MINI_H5[split]) == SPLIT_SIZES[split] for split in SPLIT_SIZES)

def select_keys(path, count, seed):
    with h5py.File(path, 'r') as h5f:
        keys = sorted(h5f.keys())
    if count > len(keys):
        raise ValueError(f'Requested {count:,} keys from {path}, only {len(keys):,} exist')
    return sorted(random.Random(seed).sample(keys, count))

def copy_split(split, selected_keys):
    local_path = LOCAL_MINI_H5[split]
    if local_path.exists():
        local_path.unlink()
    with h5py.File(SOURCE_H5[split], 'r') as src, h5py.File(local_path, 'w') as dst:
        for key in tqdm(selected_keys, desc=f'Copy {split}', unit='image'):
            src.copy(key, dst, name=key)
    with h5py.File(SOURCE_H5[split], 'r') as src, h5py.File(local_path, 'r') as dst:
        first_key = selected_keys[0]
        assert len(dst.keys()) == len(selected_keys)
        assert dst[first_key].shape == src[first_key].shape
        assert dst[first_key].dtype == src[first_key].dtype
        print(split, 'keys:', len(dst.keys()), 'sample:', dst[first_key].shape, dst[first_key].dtype)

if existing_outputs_are_valid():
    print('Mini H5 files and manifest already match this preset; skipping rebuild.')
else:
    selected = {
        'train': select_keys(SOURCE_H5['train'], TRAIN_IMAGES, SEED),
        'val': select_keys(SOURCE_H5['val'], VAL_IMAGES, SEED + 1),
    }
    for split, keys in selected.items():
        copy_split(split, keys)
    manifest = {
        'seed': SEED,
        'split_sizes': SPLIT_SIZES,
        'source_h5': {split: str(path) for split, path in SOURCE_H5.items()},
        'mini_h5': {split: str(path) for split, path in DRIVE_MINI_H5.items()},
        'selected_image_ids': selected,
    }
    local_manifest = LOCAL_OUT / MANIFEST_PATH.name
    local_manifest.write_text(json.dumps(manifest, indent=2))
    for split, path in DRIVE_MINI_H5.items():
        shutil.copy2(LOCAL_MINI_H5[split], path)
    shutil.copy2(local_manifest, MANIFEST_PATH)
    assert existing_outputs_are_valid()
    print('Mini cache synced to Drive and verified.')